# 024 — Training `unet_nll` with different beta values

Every run for the `unet_nll` model uses `loss_name="laplace_nll"`, only `beta` values differs: this notebook sweeps the **beta exponent** of the Laplace NLL

**What beta does.**  
The loss is `stop_gradient(b)^beta * (|y - mu| / b + log b)`.  
Because the weight is detached and strictly positive, beta don't move the optimum for `b` (still `b* = |y - mu|`), it rescales each pixel's contribution.  
On the gradient reaching `mu` it acts as `1 / b^(1 - beta)`: 
- `beta = 0` gives the plain Laplace NLL (`1 / b`, high-uncertainty regions downweighted most)
- `beta = 1` removes the weighting entirely

**Grid**. 
- `beta` values used are: `0.0, 0.25, 0.75`.  
- `beta = 0.5` is the default in settings so it is not retrained.  
- `beta = 1.0` is left out as the degenerate case (no weighting on `mu` at all).


Imports


In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset split
**Split used project-wide:**  
- Real **artworks** are grouped and kept entirely within one fold, exactly as a plain grouped split would do, so no painting leaks across train/val/test.  
- The **mockup** groups (listed in `settings.MOCKUP_ARTWORK_IDS`) exist purely to be learned from. They are split at the individual-pair level, with only `settings.MOCKUP_TEST_RATIO` (default 5%) held out for test. 


In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Train one run per beta

Each beta trains in its own subprocess*.  
Checkpoints and histories go to `models/beta_sweep/beta_<value>/unet_nll/`, one directory per beta value.

In [ ]:
import json
import subprocess

ARCH = "unet_nll"
LOSS_NAME = "laplace_nll"
BETAS = [0.0, 0.25, 0.75]  # 0.5 already trained -> models/nll/unet_nll/
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
SWEEP_DIR = settings.MODELS_DIR / "beta_sweep"
SWEEP_LOG_DIR = settings.LOGS_DIR / "beta_sweep"

# The beta=0.5 reference, trained by 021 — read, never retrained.
REFERENCE_BETA = settings.NLL_BETA
REFERENCE_DIR = settings.MODELS_DIR / "nll"


def run_dir(root: Path, beta: float) -> Path:
    """``<root>/beta_<value>`` — one directory per sweep point."""
    return root / f"beta_{beta:.2f}"


histories: dict[float, dict] = {}

for beta in BETAS:
    model_dir = run_dir(SWEEP_DIR, beta)
    cmd = [
        sys.executable,
        "-m",
        "scripts.train_single",
        "--arch",
        ARCH,
        "--epochs",
        str(EPOCHS),
        "--model-dir",
        str(model_dir),
        "--log-dir",
        str(run_dir(SWEEP_LOG_DIR, beta)),
        "--nll",
        "--loss-name",
        LOSS_NAME,
        "--nll-beta",
        str(beta),
    ]
    subprocess.run(cmd, cwd=project_root, check=True)

    history_path = model_dir / ARCH / "history.json"
    histories[beta] = json.loads(history_path.read_text())

    best_val_loss = min(histories[beta]["val_loss"])
    print(f"\nBest val_loss ({ARCH}, beta={beta}): {best_val_loss:.4f}")


## 3. Get the existing `beta = 0.5` train

`021_training_nll.ipynb` already trained this architecture with `beta = 0.5`-

In [ ]:
reference_history_path = REFERENCE_DIR / ARCH / "history.json"

if reference_history_path.exists():
    histories[REFERENCE_BETA] = json.loads(reference_history_path.read_text())
    print(f"Reference run loaded: beta={REFERENCE_BETA} <- {reference_history_path}")
else:
    print(
        f"No reference run at {reference_history_path} — "
        f"sweep has {len(BETAS)} points"
    )

sweep_betas = sorted(histories)
print(f"Betas available: {sweep_betas}")


## 4. Training curves

In [ ]:
for beta in sweep_betas:
    plot_training_curves(
        histories[beta], title=f"Training history — {ARCH} ({LOSS_NAME}, beta={beta})"
    )
    plt.show()


## 6. Summary

In [ ]:
for beta in sweep_betas:
    root = REFERENCE_DIR if beta == REFERENCE_BETA else run_dir(SWEEP_DIR, beta)
    ckpt = root / ARCH / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    mae_hist = histories[beta].get("val_mae")
    best_mae = f"{min(mae_hist):.4f}" if mae_hist else "n/a"
    tag = " (reference, from 021)" if beta == REFERENCE_BETA else ""
    print(f"beta={beta:<5} {status:<8} val_mae={best_mae:<8} {ckpt}{tag}")
